# Track 02.2 — Vector Indexing & Approximate Nearest Neighbor (ANN) Search

> **Prerequisites**: Track 02.1 (*Dense Embeddings, Vector Spaces, and Distance Metrics*). Familiarity with vector metrics ($L_2$, dot product, cosine similarity) and $O(N \cdot D)$ batch search.  
> **Target Audience**: AI Engineers and RAG Architects designing production-grade vector search pipelines who need to understand why exact search fails at scale and how indexes like **IVF** and **HNSW** achieve sub-millisecond retrieval.  
> **Execution Mode**: 100% Offline, CPU-only, deterministic execution with NumPy and Matplotlib.

---

### Conceptual Architecture: The Vector Indexing Landscape

```
+---------------------------------------------------------------------------------------------------+
|                                     THE VECTOR SEARCH SPECTRUM                                     |
+---------------------------------------------------------------------------------------------------+
|                                                                                                   |
|  [ Flat / Exact Scan ]                    [ Inverted File (IVF) ]              [ HNSW Multi-Layer ]   |
|  * O(N * D) Comparisons                   * Voronoi Quantization               * Skip-List on Graphs  |
|  * 100% Recall (Ground Truth)              * Query routing to nprobe clusters   * O(log N) Traversal   |
|  * Zero Memory Overhead                   * Misses vectors at cell boundaries  * Memory Heavy (links) |
|  * Latency scales linearly with N         * Fast, moderate recall              * Ultra-fast, >95% rec |
|                                                                                                   |
|  Low Scale (N < 50k) -----------------------------------------------------> Massive Scale (N > 1M)|
+---------------------------------------------------------------------------------------------------+
```

---

### Core Learning Objectives
By completing this notebook, you will:
1. **Analyze Exact $k$-NN Bottlenecks**: Quantify the computational FLOP and memory bandwidth ceiling of brute-force matrix multiplication at scale ($10^5$ to $10^8$ vectors).
2. **Formulate the ANN Pareto Frontier**: Define the fundamental trade-off space governing search: $\text{Recall}@k$ vs. Latency (QPS) vs. Memory Footprint vs. Build Index Time.
3. **Deconstruct Inverted File (IVF) Indexing**: Build an educational IVF prototype with $k$-means Voronoi partitioning, implement posting-list query routing, and visualize the **Voronoi Boundary Miss** phenomenon.
4. **Master Hierarchical Navigable Small World (HNSW)**: Understand the mathematical jump from 1D skip-lists to multi-layer metric graphs, implement an educational HNSW with geometric layer decay, greedy routing, and bounded beam search.
5. **Empirically Benchmark Trade-offs**: Directly plot the Pareto frontier (Recall@$k$ vs. Distance Comparisons) across Flat, IVF ($nprobe$ tuning), and HNSW ($efSearch$ tuning).
6. **Compare Production Vector Engines**: Understand the engineering reality behind FAISS, ChromaDB, Qdrant, and Milvus, including Product Quantization (PQ) and graph storage footprints.


## 0. Environment Setup & Configuration

We configure our Python environment using standard library utilities and pure NumPy. To prevent environment warnings across platforms, we route Matplotlib caching to `/tmp/matplotlib` and fix deterministic pseudo-random seeds (`seed=42`).


In [ ]:
import os
import sys
import time
import heapq
import collections
import warnings
from typing import List, Tuple, Dict, Set, Optional

# Ensure reproducible cache & headless rendering before importing matplotlib
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import matplotlib
matplotlib.use("Agg")  # Non-interactive headless backend for seamless headless/CI execution
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

# Master random seed for deterministic educational execution
MASTER_SEED = 42
np.random.seed(MASTER_SEED)

print("Environment initialized successfully:")
print(f"  Python version : {sys.version.split()[0]}")
print(f"  NumPy version  : {np.__version__}")


## 1. The Exact $k$-NN Baseline (Flat Index) & Algorithmic Complexity

In vector search, the **Exact Nearest Neighbor** problem is formally stated as follows:

$$\mathcal{R}_{\text{exact}}(q, k) = \operatorname{arg\,min}_{S \subset X, |S|=k} \sum_{x_i \in S} \|q - x_i\|_2$$

Given a query vector $q \in \mathbb{R}^D$ and a document matrix $X = \{x_1, x_2, \dots, x_N\} \in \mathbb{R}^{N \times D}$, an exact scan computes the distance between $q$ and every single point $x_i$:

- **Time Complexity**: $O(N \cdot D)$ distance operations per query.
- **Space Complexity**: $O(N \cdot D)$ (only storing the raw vectors).
- **Recall**: Identically $1.0$ (100% ground-truth accuracy).

Below, we implement `exact_knn_search` using vector subtraction and Euclidean norm calculation.


In [ ]:
def exact_knn_search(query: np.ndarray, dataset: np.ndarray, k: int = 5) -> List[Tuple[int, float]]:
    """Computes exact Euclidean distances from query to all vectors in dataset.
    
    Returns:
        Sorted list of (vector_id, distance) tuples for the top-k nearest neighbors.
    """
    # Pairwise difference: shape (N, D)
    diffs = dataset - query
    # L2 distance: shape (N,)
    distances = np.linalg.norm(diffs, axis=1)
    # Get top-k smallest distance indices
    top_k_indices = np.argsort(distances)[:k]
    return [(int(idx), float(distances[idx])) for idx in top_k_indices]

# Test on a small deterministic 2D dataset
sample_dataset = np.array([
    [1.0, 1.2],
    [1.1, 0.9],
    [5.0, 5.2],
    [5.2, 4.8],
    [9.0, 9.1]
], dtype=np.float64)

sample_query = np.array([1.0, 1.0], dtype=np.float64)
exact_matches = exact_knn_search(sample_query, sample_dataset, k=2)

print("Exact k-NN Query Results (k=2):")
for rank, (v_id, dist) in enumerate(exact_matches, 1):
    print(f"  Rank {rank}: Vector ID {v_id} -> Point {sample_dataset[v_id]} | L2 Distance = {dist:.4f}")


## 2. The Scaling Bottleneck & The Approximate Nearest Neighbor (ANN) Trade-off Space

Why can't production RAG systems simply use brute-force exact search for every request?

Let us calculate the concrete computational ceiling:
- Suppose a corpus contains $N = 10,000,000$ chunks embedded at dimension $D = 1536$ (OpenAI `text-embedding-3-large`).
- Each query requires:
  $$\text{FLOPs} = 10^7 \times 1536 \times 3 \approx 4.6 \times 10^{10} \text{ operations}$$
- Vector matrix size in RAM: $10^7 \times 1536 \times 4\text{ bytes} \approx 61.44\text{ GB}$.
- Under PCIe/RAM bandwidth of $100\text{ GB/s}$, simply streaming the matrix to CPU/GPU cores takes $>600\text{ ms}$ per query!

```
               THE ANN PARETO TRADEOFF FRONTIER
      Recall@k (%)
        100% +---------------------------------+  (Exact Scan: 100% Recall, High Latency)
             |                      * * * * *  |
             |                 *               |
             |             *                   |  (HNSW: High Recall, Low Latency, High RAM)
             |         *                       |
             |     *                           |  (IVF: Moderate Recall, Low RAM, Tunable)
             | *                               |
          0% +---------------------------------+
             0ms                             100ms
                          Query Latency
```

### Measuring Retrieval Accuracy: Recall@$k$
In Approximate Nearest Neighbor (ANN) search, we sacrifice exact correctness in exchange for logarithmic or sub-linear query time. We evaluate the quality of an ANN index using **Recall@$k$**:

$$\text{Recall}@k = \frac{|\mathcal{R}_{\text{ANN}}(q, k) \cap \mathcal{R}_{\text{Exact}}(q, k)|}{k}$$

Where $\mathcal{R}_{\text{ANN}}$ is the set of IDs returned by the approximate index, and $\mathcal{R}_{\text{Exact}}$ is the true top-$k$ ground truth.


In [ ]:
def compute_recall_at_k(retrieved_ids: List[int], ground_truth_ids: List[int]) -> float:
    """Computes Recall@k: fraction of true top-k nearest neighbors retrieved by ANN."""
    if not ground_truth_ids:
        return 0.0
    k = len(ground_truth_ids)
    hits = len(set(retrieved_ids[:k]).intersection(set(ground_truth_ids)))
    return float(hits / k)

# Demonstrate Recall metric behavior
ground_truth = [10, 25, 42, 99, 105]
candidate_retrieval_a = [10, 25, 42, 99, 105]  # 100% recall
candidate_retrieval_b = [10, 25, 42, 99, 777]  # 80% recall (missed 105)
candidate_retrieval_c = [1, 2, 3, 4, 5]        # 0% recall

print(f"Recall@5 (Perfect Match)  : {compute_recall_at_k(candidate_retrieval_a, ground_truth):.1%}")
print(f"Recall@5 (1 Boundary Miss): {compute_recall_at_k(candidate_retrieval_b, ground_truth):.1%}")
print(f"Recall@5 (Complete Miss)  : {compute_recall_at_k(candidate_retrieval_c, ground_truth):.1%}")


## 3. Inverted File Indexing (IVF) & Voronoi Space Partitioning

The **Inverted File (IVF)** index borrows intuition from search engine inverted indexes:

1. **Quantization / Clustering**: Partition the $D$-dimensional vector space into $C$ distinct **Voronoi cells** (clusters) by fitting $k$-means centroids: $\mu_1, \mu_2, \dots, \mu_C$.
2. **Posting Lists (Buckets)**: Assign each vector $x_i$ to its nearest centroid $\mu_c$. Store vector IDs in an inverted list (posting list) for cell $c$.
3. **Query Routing**:
   - Given query $q$, find the $nprobe$ nearest centroids to $q$.
   - Only scan vectors inside the posting lists of those $nprobe$ centroids.
   - If $nprobe = 1$, we scan roughly $N / C$ vectors ($C$-fold speedup).
   - If $nprobe = C$, we scan all vectors, degrading back to exact search.

```
       IVF SEARCH ROUTING & THE VORONOI BOUNDARY MISS
       
          Centroid A              Centroid B
          [Posting List A]        [Posting List B]
             *   *                     *    *
            *   (x1)  |              *    *
               *      |  q            *   *
                      |  * (x2)
                      |
              Cell A  |  Cell B
                      |
   * If nprobe = 1: Query q routes to Centroid B.
   * But true nearest neighbor x1 is in Cell A!
   * Result: Boundary Miss! Increasing nprobe to 2 recovers x1.
```

Below is an educational implementation of `EducationalIVFIndex`.


In [ ]:
class EducationalIVFIndex:
    """Inverted File Index (IVF) utilizing k-means clustering for Voronoi cell partitioning."""

    def __init__(self, n_clusters: int = 4, max_iters: int = 20, seed: int = 42):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.seed = seed
        self.centroids: Optional[np.ndarray] = None
        self.posting_lists: Dict[int, List[int]] = {}
        self.vectors: Optional[np.ndarray] = None

    def fit(self, vectors: np.ndarray):
        """Clusters vectors into Voronoi cells using Lloyd's k-means algorithm."""
        self.vectors = np.asarray(vectors, dtype=np.float64)
        n_samples, dim = self.vectors.shape
        rng = np.random.RandomState(self.seed)

        # 1. Random centroid initialization
        init_indices = rng.choice(n_samples, size=self.n_clusters, replace=False)
        self.centroids = self.vectors[init_indices].copy()

        # 2. Iterative Lloyd's update
        assignments = np.zeros(n_samples, dtype=int)
        for iteration in range(self.max_iters):
            # Compute distance from all points to all centroids: shape (N, C)
            dists = np.linalg.norm(self.vectors[:, None, :] - self.centroids[None, :, :], axis=2)
            new_assignments = np.argmin(dists, axis=1)

            if np.array_equal(assignments, new_assignments):
                break
            assignments = new_assignments

            # Recompute cluster centroids
            for c in range(self.n_clusters):
                cluster_pts = self.vectors[assignments == c]
                if len(cluster_pts) > 0:
                    self.centroids[c] = np.mean(cluster_pts, axis=0)

        # 3. Build inverted posting lists
        self.posting_lists = {c: [] for c in range(self.n_clusters)}
        for vec_id, c in enumerate(assignments):
            self.posting_lists[c].append(vec_id)

    def search(self, query: np.ndarray, k: int = 5, n_probe: int = 1) -> Tuple[List[Tuple[int, float]], int]:
        """Performs IVF query routing across top n_probe Voronoi cells.
        
        Returns:
            (top_k_results, total_distance_computations)
        """
        if self.centroids is None or not self.posting_lists:
            return [], 0

        # Step 1: Find n_probe nearest centroids to query
        centroid_dists = np.linalg.norm(self.centroids - query, axis=1)
        probed_clusters = np.argsort(centroid_dists)[: min(n_probe, self.n_clusters)]

        # Step 2: Gather candidate vectors from probed posting lists
        candidate_ids = []
        for c in probed_clusters:
            candidate_ids.extend(self.posting_lists[c])

        dist_computations = len(self.centroids) + len(candidate_ids)
        if not candidate_ids:
            return [], dist_computations

        # Step 3: Exact distance scan ONLY on candidate subset
        candidate_vecs = self.vectors[candidate_ids]
        candidate_dists = np.linalg.norm(candidate_vecs - query, axis=1)

        sorted_order = np.argsort(candidate_dists)[:k]
        results = [(int(candidate_ids[idx]), float(candidate_dists[idx])) for idx in sorted_order]
        return results, dist_computations

# Instantiate and fit IVF index on deterministic 2D data
rng = np.random.RandomState(MASTER_SEED)
dataset_2d = rng.randn(120, 2) * 2.5
ivf_index = EducationalIVFIndex(n_clusters=6, max_iters=25, seed=MASTER_SEED)
ivf_index.fit(dataset_2d)

print(f"IVF Index Built with {ivf_index.n_clusters} Voronoi cells across {len(dataset_2d)} points:")
for c, plist in ivf_index.posting_lists.items():
    print(f"  Cluster {c}: {len(plist)} vectors assigned (Centroid: {ivf_index.centroids[c].round(2)})")


## 4. Visualizing IVF Voronoi Cells & The Boundary Miss Phenomenon

To build concrete intuition for how Voronoi partitioning works and why boundary misses occur, we construct a 2D meshgrid to visualize:
1. The **Voronoi cell regions** separated by partition hyperplanes.
2. The **Centroids** defining the cell centers.
3. A **Query point** placed intentionally near a boundary line.
4. The behavior of $nprobe=1$ (misses the neighbor on the opposite side of the line) versus $nprobe=2$ (successfully retrieves the true nearest neighbor).


In [ ]:
# Set up 2D visualization grid
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

x_min, x_max = dataset_2d[:, 0].min() - 1, dataset_2d[:, 0].max() + 1
y_min, y_max = dataset_2d[:, 1].min() - 1, dataset_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Assign grid points to nearest centroid to plot Voronoi regions
grid_dists = np.linalg.norm(grid_points[:, None, :] - ivf_index.centroids[None, :, :], axis=2)
grid_cells = np.argmin(grid_dists, axis=1).reshape(xx.shape)

# Pick an intentional query near a boundary
test_query = np.array([0.2, 0.3])
exact_gt = exact_knn_search(test_query, dataset_2d, k=3)
gt_ids = [idx for idx, _ in exact_gt]

for ax, n_probe in zip([ax1, ax2], [1, 2]):
    # Draw Voronoi decision boundary background
    ax.contourf(xx, yy, grid_cells, alpha=0.15, cmap='tab10')
    ax.contour(xx, yy, grid_cells, colors='gray', linewidths=0.6, linestyles='--')

    # Draw dataset points
    for c, p_list in ivf_index.posting_lists.items():
        pts = dataset_2d[p_list]
        ax.scatter(pts[:, 0], pts[:, 1], s=25, alpha=0.6, label=f"Cluster {c}" if n_probe == 1 else "")

    # Draw centroids
    ax.scatter(ivf_index.centroids[:, 0], ivf_index.centroids[:, 1], 
               c='black', marker='X', s=90, label="Centroids" if n_probe == 1 else "")

    # Run IVF search
    ivf_res, comps = ivf_index.search(test_query, k=3, n_probe=n_probe)
    retrieved_ids = [idx for idx, _ in ivf_res]
    recall = compute_recall_at_k(retrieved_ids, gt_ids)

    # Highlight Query point
    ax.scatter(test_query[0], test_query[1], c='red', marker='*', s=220, edgecolors='black', label="Query", zorder=5)

    # Highlight Retrieved Points
    if retrieved_ids:
        ret_pts = dataset_2d[retrieved_ids]
        ax.scatter(ret_pts[:, 0], ret_pts[:, 1], s=110, facecolors='none', 
                   edgecolors='red', linewidths=2.0, label="Retrieved", zorder=4)

    # Highlight Ground Truth Points
    gt_pts = dataset_2d[gt_ids]
    ax.scatter(gt_pts[:, 0], gt_pts[:, 1], s=160, facecolors='none', 
               edgecolors='green', linewidths=1.5, linestyle=':', label="Ground Truth", zorder=4)

    ax.set_title(f"IVF Search (nprobe={n_probe}) | Recall@3 = {recall:.0%} | Dist Calcs = {comps}")
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.grid(True, linestyle=":", alpha=0.4)

ax1.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


## 5. Hierarchical Navigable Small World (HNSW) Graphs

While IVF reduces candidate comparisons via space quantization, **Hierarchical Navigable Small World (HNSW)** (Malkov & Yashunin, 2018) is the industry gold standard across vector search engines (ChromaDB, Milvus, Qdrant, Pinecone).

### 1. From 1D Skip-Lists to Metric Proximity Graphs
William Pugh (1990) introduced the **Skip-List**: an ordinary linked list with $O(N)$ sequential traversal is accelerated to $O(\log N)$ by constructing layered "express lanes" where higher layers skip over large blocks of elements.

```
  Layer 2 (Express): [ 0 ] ----------------------------------------------> [ 8 ]
  Layer 1 (Medium) : [ 0 ] -----------------> [ 4 ] ---------------------> [ 8 ]
  Layer 0 (Base)   : [ 0 ] -> [ 1 ] -> [ 2 ] -> [ 4 ] -> [ 5 ] -> [ 7 ] -> [ 8 ]
```

HNSW generalizes this exact hierarchy to multidimensional metric spaces:
- **Layer 0**: Contains **every vector in the dataset**. Graph edges are dense and short-range.
- **Higher Layers ($l > 0$)**: Contain exponentially fewer vectors with long-range "highway" edges spanning across distant clusters.

### 2. Probabilistic Layer Assignment: Geometric Decay
When a new vector is inserted, its maximum level $l$ is chosen randomly from a geometric distribution parameterized by $m_L = 1 / \ln(M)$:

$$l = \left\lfloor -\ln(\text{uniform}(0, 1)) \cdot m_L \right\rfloor$$

This guarantees that:
- $P(\text{level} \ge 0) = 1.0$ (100% of nodes in Layer 0)
- $P(\text{level} \ge 1) \approx 1 / M$
- $P(\text{level} \ge 2) \approx 1 / M^2$

### 3. The Search Algorithm (Greedy Descent + Layer 0 Beam Search)
1. **Hierarchical Descent ($l = L_{\text{max}} \dots 1$)**: Start at the global entry point $ep$ at the top layer. Greedily hop to whichever connected neighbor is closest to query $q$. When reaching a local minimum at layer $l$, drop that node as the entry point to layer $l-1$.
2. **Base Layer Beam Search ($l = 0$)**: Perform bounded priority queue search (`efSearch`) to explore the local neighborhood and return the top-$k$ nearest neighbors.

Below is our pure-Python educational `EducationalHNSWIndex`.


In [ ]:
class EducationalHNSWIndex:
    """Educational Hierarchical Navigable Small World (HNSW) index in pure Python/NumPy."""

    def __init__(self, dim: int, m: int = 6, ef_construction: int = 16, seed: int = 42):
        self.dim = dim
        self.m = m                 # Max edges per node in layers > 0
        self.m0 = 2 * m            # Max edges per node in base layer 0
        self.ef_construction = ef_construction
        self.ml = 1.0 / np.log(m)  # Level generation multiplier
        self.rng = np.random.RandomState(seed)

        self.vectors: Dict[int, np.ndarray] = {}
        self.levels: Dict[int, int] = {}
        self.graphs: List[Dict[int, Set[int]]] = []  # graphs[level][node_id] = set of neighbor_ids
        self.enter_point: Optional[int] = None
        self.max_level: int = -1

    def _dist(self, u: np.ndarray, v: np.ndarray) -> float:
        """Euclidean distance between two vectors."""
        return float(np.linalg.norm(u - v))

    def _get_random_level(self) -> int:
        """Sample level using exponential decay: floor(-ln(uniform) * ml)."""
        r = self.rng.uniform(0.0001, 1.0)
        return int(-np.log(r) * self.ml)

    def insert(self, vec_id: int, vec: np.ndarray):
        """Inserts a vector into the HNSW multi-layer graph."""
        vec = np.asarray(vec, dtype=np.float64)
        self.vectors[vec_id] = vec
        level = self._get_random_level()
        self.levels[vec_id] = level

        # Ensure graph levels exist
        while len(self.graphs) <= max(level, self.max_level):
            self.graphs.append({})
        for l in range(level + 1):
            if vec_id not in self.graphs[l]:
                self.graphs[l][vec_id] = set()

        # If index is empty, new node becomes global entry point
        if self.enter_point is None:
            self.enter_point = vec_id
            self.max_level = level
            return

        curr_obj = self.enter_point

        # Phase 1: Greedy routing through top layers down to level + 1
        for l in range(self.max_level, level, -1):
            changed = True
            while changed:
                changed = False
                curr_dist = self._dist(vec, self.vectors[curr_obj])
                for n_id in self.graphs[l].get(curr_obj, set()):
                    d = self._dist(vec, self.vectors[n_id])
                    if d < curr_dist:
                        curr_dist = d
                        curr_obj = n_id
                        changed = True

        # Phase 2: Insert into layers from min(max_level, level) down to 0
        ep = [curr_obj]
        for l in range(min(self.max_level, level), -1, -1):
            # Bounded search to find closest candidates at level l
            v = set(ep)
            candidates = []
            results = []
            for node in ep:
                d = self._dist(vec, self.vectors[node])
                heapq.heappush(candidates, (d, node))
                heapq.heappush(results, (-d, node))

            while candidates:
                c_dist, c_id = heapq.heappop(candidates)
                furthest = -results[0][0]
                if c_dist > furthest:
                    break
                for n_id in self.graphs[l].get(c_id, set()):
                    if n_id not in v:
                        v.add(n_id)
                        n_dist = self._dist(vec, self.vectors[n_id])
                        furthest = -results[0][0]
                        if n_dist < furthest or len(results) < self.ef_construction:
                            heapq.heappush(candidates, (n_dist, n_id))
                            heapq.heappush(results, (-n_dist, n_id))
                            if len(results) > self.ef_construction:
                                heapq.heappop(results)

            ep = [item[1] for item in sorted([(-d, nid) for d, nid in results])]
            m_max = self.m0 if l == 0 else self.m

            # Connect new node to nearest neighbors
            neighbors = [n_id for n_id in ep[:m_max] if n_id != vec_id]
            self.graphs[l][vec_id] = set(neighbors)
            for n_id in neighbors:
                self.graphs[l][n_id].add(vec_id)
                # Shrink neighbor list if exceeding degree limit
                if len(self.graphs[l][n_id]) > m_max:
                    n_nbrs = list(self.graphs[l][n_id])
                    n_dists = [self._dist(self.vectors[n_id], self.vectors[other]) for other in n_nbrs]
                    sorted_nbrs = [x for _, x in sorted(zip(n_dists, n_nbrs))]
                    self.graphs[l][n_id] = set(sorted_nbrs[:m_max])

        # Update global entry point if new node has higher level
        if level > self.max_level:
            self.max_level = level
            self.enter_point = vec_id

    def search(self, query: np.ndarray, k: int = 5, ef_search: int = 16) -> Tuple[List[Tuple[int, float]], int]:
        """Executes HNSW search: greedy routing down to Layer 0, then beam search."""
        if self.enter_point is None:
            return [], 0

        query = np.asarray(query, dtype=np.float64)
        dist_comps = 0
        dist_cache: Dict[int, float] = {}

        def get_dist(node_id: int) -> float:
            nonlocal dist_comps
            if node_id not in dist_cache:
                dist_comps += 1
                dist_cache[node_id] = self._dist(query, self.vectors[node_id])
            return dist_cache[node_id]

        curr_obj = self.enter_point

        # Step 1: Greedy traversal in upper layers
        for l in range(self.max_level, 0, -1):
            changed = True
            while changed:
                changed = False
                curr_dist = get_dist(curr_obj)
                for n_id in self.graphs[l].get(curr_obj, set()):
                    d = get_dist(n_id)
                    if d < curr_dist:
                        curr_dist = d
                        curr_obj = n_id
                        changed = True

        # Step 2: Layer 0 Priority-Queue Beam Search
        ef = max(ef_search, k)
        visited = {curr_obj}
        candidates = []
        results = []

        d0 = get_dist(curr_obj)
        heapq.heappush(candidates, (d0, curr_obj))
        heapq.heappush(results, (-d0, curr_obj))

        while candidates:
            c_dist, c_id = heapq.heappop(candidates)
            furthest = -results[0][0]
            if c_dist > furthest:
                break
            for n_id in self.graphs[0].get(c_id, set()):
                if n_id not in visited:
                    visited.add(n_id)
                    n_dist = get_dist(n_id)
                    furthest = -results[0][0]
                    if n_dist < furthest or len(results) < ef:
                        heapq.heappush(candidates, (n_dist, n_id))
                        heapq.heappush(results, (-n_dist, n_id))
                        if len(results) > ef:
                            heapq.heappop(results)

        top_k = sorted([(n_id, -d) for d, n_id in results], key=lambda x: x[1])[:k]
        return top_k, dist_comps

# Build HNSW index on our 2D dataset
hnsw_index = EducationalHNSWIndex(dim=2, m=4, ef_construction=16, seed=MASTER_SEED)
for i, pt in enumerate(dataset_2d):
    hnsw_index.insert(i, pt)

print("HNSW Index Built:")
print(f"  Total Nodes Inserted: {len(hnsw_index.vectors)}")
print(f"  Max Level Reached    : {hnsw_index.max_level}")
print(f"  Global Entry Point   : Node {hnsw_index.enter_point}")
for l in range(len(hnsw_index.graphs)):
    print(f"  Layer {l} Node Count  : {len(hnsw_index.graphs[l])} nodes")


## 6. Visualizing HNSW Multi-Layer Skip Graph & Search Trajectory

To understand how HNSW navigates through space, we plot each layer of our index:
- **Layer 2 (Top Express Lane)**: Sparse nodes with long-range bridges.
- **Layer 1 (Intermediate Layer)**: Moderate density narrowing down toward the target region.
- **Layer 0 (Base Graph)**: Complete dataset with dense clustering connections.
- **Query Trajectory**: The sequence of hops taken by the query from the global entry point down to the final top-$k$ nearest neighbors.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
levels_to_plot = [0, 1, min(2, hnsw_index.max_level)]

for ax, lvl in zip(axes, [0, 1, 2]):
    if lvl > hnsw_index.max_level:
        ax.set_title(f"Layer {lvl} (Empty)")
        continue

    # Plot all graph edges at this layer
    for u, neighbors in hnsw_index.graphs[lvl].items():
        pu = dataset_2d[u]
        for v in neighbors:
            if u < v:  # Avoid double plotting undirected edges
                pv = dataset_2d[v]
                ax.plot([pu[0], pv[0]], [pu[1], pv[1]], c='lightblue' if lvl == 0 else 'royalblue', 
                        alpha=0.4 if lvl == 0 else 0.7, linewidth=0.8 if lvl == 0 else 1.2)

    # Plot nodes present at this layer
    nodes = list(hnsw_index.graphs[lvl].keys())
    node_pts = dataset_2d[nodes]
    ax.scatter(node_pts[:, 0], node_pts[:, 1], c='navy' if lvl > 0 else 'steelblue', 
               s=20 if lvl == 0 else 55, alpha=0.8, zorder=3)

    # Highlight entry point at top layer
    if lvl == hnsw_index.max_level:
        ep_pt = dataset_2d[hnsw_index.enter_point]
        ax.scatter(ep_pt[0], ep_pt[1], c='gold', marker='P', s=160, edgecolors='black', label="Entry Point", zorder=5)

    # Plot query
    ax.scatter(test_query[0], test_query[1], c='red', marker='*', s=180, edgecolors='black', label="Query", zorder=5)

    ax.set_title(f"HNSW Layer {lvl} ({len(nodes)} nodes)")
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.grid(True, linestyle=":", alpha=0.4)
    if lvl == hnsw_index.max_level:
        ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()


## 7. Empirical Trade-off Analysis: Exact vs. IVF vs. HNSW Pareto Frontier

Now we perform a controlled, quantitative benchmark across a 16-dimensional dataset ($N = 250$, $D = 16$).

We evaluate:
1. **Exact Flat Scan**: Ground-truth baseline requiring $N = 250$ distance calculations per query.
2. **IVF Index**: Sweeping $nprobe \in [1, 2, 4, 8, 12]$ across 12 Voronoi clusters.
3. **HNSW Index**: Sweeping $efSearch \in [2, 5, 10, 20, 40]$ with $M = 6$.

For each configuration, we measure:
- **Recall@5**: Average retrieval accuracy across 10 evaluation queries.
- **Distance Computations**: Total vector distance evaluations performed (proxy for query latency/computational effort).


In [ ]:
# 1. Generate controlled benchmark dataset in R^16
bench_rng = np.random.RandomState(MASTER_SEED)
N_bench, D_bench = 250, 16
X_bench = bench_rng.randn(N_bench, D_bench)
queries = bench_rng.randn(10, D_bench)
K_VAL = 5

# 2. Compute exact ground truth for all queries
ground_truths = []
for q in queries:
    gt_res = exact_knn_search(q, X_bench, k=K_VAL)
    ground_truths.append([idx for idx, _ in gt_res])

# 3. Fit IVF index
ivf_bench = EducationalIVFIndex(n_clusters=12, max_iters=25, seed=MASTER_SEED)
ivf_bench.fit(X_bench)

# 4. Build HNSW index
hnsw_bench = EducationalHNSWIndex(dim=D_bench, m=6, ef_construction=24, seed=MASTER_SEED)
for i in range(N_bench):
    hnsw_bench.insert(i, X_bench[i])

# 5. Benchmark IVF across nprobe sweeps
ivf_probes = [1, 2, 4, 6, 8, 12]
ivf_recalls, ivf_efforts = [], []
for np_val in ivf_probes:
    rec_list, effort_list = [], []
    for q, gt in zip(queries, ground_truths):
        res, comps = ivf_bench.search(q, k=K_VAL, n_probe=np_val)
        ret_ids = [idx for idx, _ in res]
        rec_list.append(compute_recall_at_k(ret_ids, gt))
        effort_list.append(comps)
    ivf_recalls.append(float(np.mean(rec_list)))
    ivf_efforts.append(float(np.mean(effort_list)))

# 6. Benchmark HNSW across efSearch sweeps
hnsw_efs = [2, 5, 10, 20, 35, 50]
hnsw_recalls, hnsw_efforts = [], []
for ef_val in hnsw_efs:
    rec_list, effort_list = [], []
    for q, gt in zip(queries, ground_truths):
        res, comps = hnsw_bench.search(q, k=K_VAL, ef_search=ef_val)
        ret_ids = [idx for idx, _ in res]
        rec_list.append(compute_recall_at_k(ret_ids, gt))
        effort_list.append(comps)
    hnsw_recalls.append(float(np.mean(rec_list)))
    hnsw_efforts.append(float(np.mean(effort_list)))

# Plot the Pareto Frontier: Recall@5 vs. Distance Comparisons
plt.figure(figsize=(9, 5.5))
plt.plot(ivf_efforts, ivf_recalls, 'o-', color='tab:orange', linewidth=2, label="IVF (nprobe sweep)")
plt.plot(hnsw_efforts, hnsw_recalls, 's-', color='tab:green', linewidth=2, label="HNSW (efSearch sweep)")
plt.axvline(x=N_bench, color='tab:red', linestyle='--', label=f"Exact Scan ({N_bench} comparisons, 100% Recall)")

# Annotate points
for x, y, p in zip(ivf_efforts, ivf_recalls, ivf_probes):
    plt.annotate(f"np={p}", (x, y), textcoords="offset points", xytext=(0, 7), ha='center', fontsize=8)
for x, y, ef in zip(hnsw_efforts, hnsw_recalls, hnsw_efs):
    plt.annotate(f"ef={ef}", (x, y), textcoords="offset points", xytext=(0, -12), ha='center', fontsize=8)

plt.title(f"The ANN Pareto Frontier: Recall@5 vs. Distance Evaluations (N={N_bench}, D={D_bench})")
plt.xlabel("Average Vector Distance Computations per Query (Lower = Faster)")
plt.ylabel("Average Recall@5 (Higher = Better Accuracy)")
plt.ylim(0.0, 1.05)
plt.grid(True, linestyle=":", alpha=0.5)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print("Pareto Benchmark Summary:")
print(f"  Exact Flat Scan : 100% Recall @ {N_bench} distance calculations")
print(f"  HNSW (ef=20)    : {hnsw_recalls[3]:.1%} Recall @ {hnsw_efforts[3]:.1f} distance calculations ({N_bench/hnsw_efforts[3]:.1f}x speedup)")
print(f"  IVF (nprobe=4)  : {ivf_recalls[2]:.1%} Recall @ {ivf_efforts[2]:.1f} distance calculations ({N_bench/ivf_efforts[2]:.1f}x speedup)")


## 8. Production Vector Database Realities: Quantization & Memory Footprint

In production vector databases like **ChromaDB**, **Milvus**, and **Qdrant**, architecture decisions depend heavily on memory footprints:

| Index Architecture | Latency @ 1M Vectors | RAM per 1M Vectors ($D=768$) | Recall@10 | Best Use Case |
|---|---|---|---|---|
| **Flat (Exact)** | $50 - 200\text{ ms}$ | $3.07\text{ GB}$ (exact float32) | $100\%$ | Small corpora ($N < 50\text{k}$), zero tolerance for missing chunks |
| **IVF-PQ (Product Quantized)** | $5 - 15\text{ ms}$ | $0.20 - 0.40\text{ GB}$ (compressed codes) | $80 - 92\%$ | Massive scale ($10^7 - 10^9$), RAM-constrained budgets |
| **HNSW (Graph Index)** | $1 - 3\text{ ms}$ | $4.5 - 6.0\text{ GB}$ (vectors + link graph) | $95 - 99\%$ | Low latency SLA (<5ms), high recall required, sufficient RAM |

### The HNSW Memory Overhead Formula
In HNSW, in addition to the raw vectors ($4 \cdot D$ bytes per vector), every node stores an adjacency list of neighbor IDs (4 bytes each) across all layers:

$$\text{Memory}_{\text{HNSW}} = N \cdot \left( 4D + 4 \cdot M_{\text{avg}} \right) \text{ bytes}$$

Where $M_{\text{avg}} \approx 2 \cdot M$ due to Layer 0 bi-directional links.
Below, we compute the RAM requirements for a 1-million vector index.


In [ ]:
def estimate_index_ram_gb(n_vectors: int, dim: int, m: int = 16, pq_bytes: int = 64) -> Dict[str, float]:
    """Calculates theoretical RAM requirements in GB across index types."""
    raw_vector_bytes = n_vectors * dim * 4  # 4 bytes for float32
    
    # Flat: only vectors
    flat_gb = raw_vector_bytes / (1024 ** 3)
    
    # HNSW: raw vectors + graph neighbor pointers (approx 2 * M per node * 4 bytes)
    hnsw_graph_bytes = n_vectors * (2 * m) * 4
    hnsw_gb = (raw_vector_bytes + hnsw_graph_bytes) / (1024 ** 3)
    
    # IVF-PQ: quantized codes (e.g., 64 bytes per vector) + centroid codebooks
    ivf_pq_gb = (n_vectors * pq_bytes) / (1024 ** 3)
    
    return {
        "Flat (Brute Force)": round(flat_gb, 2),
        "HNSW (M=16)": round(hnsw_gb, 2),
        "IVF-PQ (64 bytes/vec)": round(ivf_pq_gb, 2)
    }

ram_1m_768 = estimate_index_ram_gb(n_vectors=1_000_000, dim=768, m=16)
ram_10m_1536 = estimate_index_ram_gb(n_vectors=10_000_000, dim=1536, m=16)

print("Estimated RAM Consumption (1 Million Vectors, D=768):")
for idx_type, ram in ram_1m_768.items():
    print(f"  * {idx_type:22s}: {ram:.2f} GB")

print("\nEstimated RAM Consumption (10 Million Vectors, D=1536):")
for idx_type, ram in ram_10m_1536.items():
    print(f"  * {idx_type:22s}: {ram:.2f} GB")


## 9. Notebook Verification & Automated Validation Suite

To ensure educational reproducibility and mathematical correctness, we execute a rigorous series of in-notebook assertions:
1. **Exact $k$-NN Precision**: Verifies ordering and distance metric calculations against manual calculations.
2. **IVF Inverted List Completeness**: Asserts all $N$ vectors are partitioned with zero leaks or duplicates, and full probe ($nprobe=C$) achieves 100% recall.
3. **HNSW Skip-List Hierarchy**: Verifies layer decay distribution ($|L_0| \ge |L_1| \ge |L_2|$) and greedy routing convergence.
4. **Recall@$k$ Metric Invariants**: Confirms proper handling of perfect, partial, and empty intersections.


In [ ]:
# 1. Exact k-NN Precision Test
test_data = np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 2.0]], dtype=np.float64)
q_test = np.array([0.9, 0.9], dtype=np.float64)
exact_check = exact_knn_search(q_test, test_data, k=2)
assert exact_check[0][0] == 1, f"Expected top match index 1, got {exact_check[0][0]}"
assert exact_check[0][1] <= exact_check[1][1], "Exact results not monotonically sorted"

# 2. IVF Completeness & Full Probe Invariant
test_ivf = EducationalIVFIndex(n_clusters=3, max_iters=15, seed=42)
test_ivf.fit(test_data)
total_ivf_assigned = sum(len(plist) for plist in test_ivf.posting_lists.values())
assert total_ivf_assigned == len(test_data), "IVF failed to index all vectors"
ivf_full_res, _ = test_ivf.search(q_test, k=2, n_probe=3)
assert compute_recall_at_k([r[0] for r in ivf_full_res], [r[0] for r in exact_check]) == 1.0

# 3. HNSW Layer Hierarchy Test
test_hnsw = EducationalHNSWIndex(dim=4, m=4, ef_construction=12, seed=42)
for i in range(50):
    test_hnsw.insert(i, np.random.RandomState(i).randn(4))
assert len(test_hnsw.graphs[0]) == 50, "HNSW Layer 0 must contain all 50 nodes"
for l in range(test_hnsw.max_level):
    assert len(test_hnsw.graphs[l]) >= len(test_hnsw.graphs[l+1]), "Violated HNSW skip-list layer decay"

# 4. Recall Metric Invariants
assert compute_recall_at_k([1, 2, 3], [1, 2, 3]) == 1.0
assert compute_recall_at_k([1, 2, 4], [1, 2, 3]) == pytest.approx(2/3) if 'pytest' in sys.modules else abs(compute_recall_at_k([1, 2, 4], [1, 2, 3]) - 2/3) < 1e-6
assert compute_recall_at_k([4, 5, 6], [1, 2, 3]) == 0.0

print("All 4 in-notebook verification suites PASSED successfully!")
print("Track 02.2 (Vector Indexing & ANN Search) verified and mathematically sound.")
